# Opponent ELO & ACPL Visualization

Two data sources:
1. **PGN file** — provides opponent ratings from `WhiteElo` / `BlackElo` headers.
2. **ACPL log** (paste of `acpl_elo_estimator_sf.py` stdout) — provides per-game ACPL and estimated-ELO.

Joined on `(White, Black, Result)`. Target player is auto-detected as the name appearing in every game.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import chess.pgn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Inputs ---
PGN_PATH = Path('chess_com_games_2026-05-20.pgn')
ACPL_LOG_PATH = Path('output.txt')
BUCKET_SIZE = 100  # ELO bucket width

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Load PGN → DataFrame of games + ratings

In [ ]:
def parse_pgn(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding='utf-8') as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break
            h = game.headers
            def parse_elo(v):
                try:
                    return int(v)
                except (ValueError, TypeError):
                    return None
            rows.append({
                'white': h.get('White', ''),
                'black': h.get('Black', ''),
                'white_elo': parse_elo(h.get('WhiteElo')),
                'black_elo': parse_elo(h.get('BlackElo')),
                'result': h.get('Result', '*'),
                'date': h.get('Date', ''),
            })
    return pd.DataFrame(rows)

pgn_df = parse_pgn(PGN_PATH)
print(f'Loaded {len(pgn_df)} games from {PGN_PATH.name}')
pgn_df.head()

## 2. Parse ACPL log

Expected line format from `acpl_elo_estimator_sf.py`:
```
[Game 4/110] reyquaza67 vs POTUS_Official (0-1) -- eval as Black | moves=25 ACPL=70.4 Est.ELO=2143 (25.5s)
```

In [ ]:
ACPL_LINE_RE = re.compile(
    r'\[Game (\d+)/\d+\] (\S+) vs (\S+) \(([\d/-]+)\) '
    r'-- eval as (White|Black) \| moves=(\d+) ACPL=([\d.]+) '
    r'Est\.ELO=(\d+) \(([\d.]+)s\)'
)

def parse_acpl_log(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        m = ACPL_LINE_RE.search(line)
        if not m:
            continue
        idx, white, black, result, target, moves, acpl, est_elo, elapsed = m.groups()
        rows.append({
            'game_idx': int(idx),
            'white': white,
            'black': black,
            'result': result,
            'target_color': target,
            'moves_evaluated': int(moves),
            'acpl': float(acpl),
            'est_elo': int(est_elo),
            'elapsed_s': float(elapsed),
        })
    return pd.DataFrame(rows)

acpl_df = parse_acpl_log(ACPL_LOG_PATH)
print(f'Parsed {len(acpl_df)} ACPL records from {ACPL_LOG_PATH.name}')
acpl_df.head()

## 3. Auto-detect target player and join

In [ ]:
# Target = the name appearing in every (or nearly every) ACPL game.
name_counts = Counter()
for _, r in acpl_df.iterrows():
    name_counts[r['white']] += 1
    name_counts[r['black']] += 1
target_player, target_count = name_counts.most_common(1)[0]
print(f'Target player: {target_player} (appeared in {target_count}/{len(acpl_df)} games)')

# Join ACPL records to PGN games on (white, black, result). This is unique within
# a single-player PGN export since opponent names rarely repeat.
merged = acpl_df.merge(
    pgn_df[['white', 'black', 'result', 'white_elo', 'black_elo']],
    on=['white', 'black', 'result'],
    how='inner',
)

# Compute opponent ELO from target_color: if target played White, opponent is Black, etc.
merged['opponent_elo'] = np.where(
    merged['target_color'] == 'White', merged['black_elo'], merged['white_elo']
)
merged['target_elo'] = np.where(
    merged['target_color'] == 'White', merged['white_elo'], merged['black_elo']
)
# Target's outcome: 1 = win, 0 = loss, 0.5 = draw
def target_score(row):
    if row['result'] == '1/2-1/2':
        return 0.5
    target_won = (row['result'] == '1-0') == (row['target_color'] == 'White')
    return 1.0 if target_won else 0.0
merged['target_score'] = merged.apply(target_score, axis=1)

print(f'Joined {len(merged)} records (dropped {len(acpl_df) - len(merged)} unmatched)')
merged = merged.dropna(subset=['opponent_elo']).copy()
merged['opponent_elo'] = merged['opponent_elo'].astype(int)
print(f'After dropping rows with no opponent ELO: {len(merged)}')
merged.head()

## 4. Opponent ELO distribution (100-buckets)

In [ ]:
def bucket(series, size=BUCKET_SIZE):
    return (series // size) * size

merged['opp_bucket'] = bucket(merged['opponent_elo'])

bucket_counts = merged['opp_bucket'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(bucket_counts.index, bucket_counts.values, width=BUCKET_SIZE * 0.9,
       align='edge', edgecolor='black', alpha=0.8)
ax.set_xlabel(f'Opponent ELO (bucketed by {BUCKET_SIZE})')
ax.set_ylabel('Number of games')
ax.set_title(f'Opponent ELO distribution — {target_player} ({len(merged)} games)')
ax.set_xticks(bucket_counts.index)
ax.tick_params(axis='x', rotation=45)
for x, y in zip(bucket_counts.index, bucket_counts.values):
    ax.text(x + BUCKET_SIZE / 2, y + 0.3, str(y), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'Median opponent ELO: {int(merged["opponent_elo"].median())}')
print(f'Mean opponent ELO:   {int(merged["opponent_elo"].mean())}')
print(f'Range: {int(merged["opponent_elo"].min())}–{int(merged["opponent_elo"].max())}')

## 5. ACPL vs Opponent ELO

Scatter of every game + per-bucket mean line. **Higher-rated opponents should push the target's ACPL up** (their position is harder, more chances to drift).

In [ ]:
bucket_acpl = merged.groupby('opp_bucket').agg(
    mean_acpl=('acpl', 'mean'),
    median_acpl=('acpl', 'median'),
    n=('acpl', 'size'),
).reset_index()

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(merged['opponent_elo'], merged['acpl'],
           alpha=0.4, s=30, label='per-game ACPL')
ax.plot(bucket_acpl['opp_bucket'] + BUCKET_SIZE / 2, bucket_acpl['mean_acpl'],
        'o-', color='red', linewidth=2, markersize=8, label=f'bucket mean ({BUCKET_SIZE} ELO bins)')
ax.plot(bucket_acpl['opp_bucket'] + BUCKET_SIZE / 2, bucket_acpl['median_acpl'],
        's--', color='orange', linewidth=1.5, markersize=6, alpha=0.7, label='bucket median')
ax.set_xlabel('Opponent ELO')
ax.set_ylabel("Target's ACPL (lower = better)")
ax.set_title(f'ACPL vs Opponent ELO — {target_player}')
ax.legend()
plt.tight_layout()
plt.show()

# Correlation
corr = merged[['opponent_elo', 'acpl']].corr().iloc[0, 1]
print(f'Pearson correlation (opponent_elo, ACPL): {corr:+.3f}')
print(f'(Positive ⇒ harder opponents drive higher ACPL, as expected.)')

## 6. Win rate by opponent ELO bucket

In [ ]:
bucket_score = merged.groupby('opp_bucket').agg(
    score=('target_score', 'mean'),
    n=('target_score', 'size'),
).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#2ecc71' if s >= 0.5 else '#e74c3c' for s in bucket_score['score']]
ax.bar(bucket_score['opp_bucket'], bucket_score['score'],
       width=BUCKET_SIZE * 0.9, align='edge', color=colors,
       edgecolor='black', alpha=0.8)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel(f'Opponent ELO (bucket of {BUCKET_SIZE})')
ax.set_ylabel('Score (1 = win, 0.5 = draw, 0 = loss)')
ax.set_title(f'Outcome by opponent strength — {target_player}')
for _, row in bucket_score.iterrows():
    ax.text(row['opp_bucket'] + BUCKET_SIZE / 2, row['score'] + 0.02,
            f"{row['score']:.0%}\n(n={int(row['n'])})", ha='center', fontsize=8)
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

## 7. Estimated-ELO distribution (per-game)

How variable is the per-game Stockfish ACPL→ELO estimate? Heavy spread means individual games are unreliable indicators.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(merged['est_elo'], bins=range(0, 3400, BUCKET_SIZE),
        edgecolor='black', alpha=0.8)
ax.axvline(merged['est_elo'].mean(), color='red', linewidth=2,
           label=f"mean = {int(merged['est_elo'].mean())}")
ax.axvline(merged['est_elo'].median(), color='orange', linewidth=2, linestyle='--',
           label=f"median = {int(merged['est_elo'].median())}")
ax.set_xlabel('Per-game estimated ELO')
ax.set_ylabel('Number of games')
ax.set_title(f'Per-game estimated-ELO distribution — {target_player}')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Moves evaluated vs ACPL

Short games can have noisy ACPL (small denominator + a single blunder swings the average). Useful to see whether the high-ACPL games are systematically short.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(merged['moves_evaluated'], merged['acpl'], alpha=0.5, s=30)
ax.set_xlabel('Moves evaluated (target player only)')
ax.set_ylabel('ACPL')
ax.set_title('Game length vs ACPL — does ACPL settle with more moves?')
# LOWESS-ish smoothing via rolling mean over sorted x
sorted_pts = merged.sort_values('moves_evaluated')
window = max(5, len(sorted_pts) // 10)
rolling = sorted_pts['acpl'].rolling(window=window, center=True, min_periods=3).mean()
ax.plot(sorted_pts['moves_evaluated'], rolling, color='red',
        linewidth=2, label=f'rolling mean (window={window})')
ax.legend()
plt.tight_layout()
plt.show()

## 9. ACPL split by side (White vs Black)

Does the target play noticeably better as one color?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for color, data in merged.groupby('target_color'):
    ax1.hist(data['acpl'], bins=20, alpha=0.5, label=f'{color} (n={len(data)}, mean={data["acpl"].mean():.1f})')
ax1.set_xlabel('ACPL')
ax1.set_ylabel('Games')
ax1.set_title('ACPL distribution by side')
ax1.legend()

# Box plot for cleaner side-by-side
white_acpl = merged[merged['target_color'] == 'White']['acpl']
black_acpl = merged[merged['target_color'] == 'Black']['acpl']
ax2.boxplot([white_acpl, black_acpl], labels=['White', 'Black'])
ax2.set_ylabel('ACPL')
ax2.set_title('ACPL by side (box plot)')

plt.tight_layout()
plt.show()

## 10. Summary table

In [ ]:
summary = merged.groupby('opp_bucket').agg(
    n=('acpl', 'size'),
    mean_acpl=('acpl', 'mean'),
    median_acpl=('acpl', 'median'),
    mean_est_elo=('est_elo', 'mean'),
    score=('target_score', 'mean'),
).round(1)
summary['score'] = summary['score'].apply(lambda x: f'{x:.0%}')
summary.index = summary.index.map(lambda b: f'{int(b)}-{int(b) + BUCKET_SIZE - 1}')
summary.index.name = 'opponent_elo_bucket'
summary